#### https://github.com/HVL-ML/DAT255/blob/main/notebooks/15_gradio_and_streamlit.ipynb er brukt som utgangspunkt i denne notebooken.

# Gradio (and Streamlit) deployment

For deploying an ML-based app there are various approaches, but [Gradio](gradio.app) and [Streamlit](https://streamlit.io/) are both quick and convenient ways to do so. Typically we would implement this in a plain python (`.py`) file rather than a `.ipynb` notebook, but Gradio works here too, so let's try that first. Streamlit needs to be run directly in python, but the code is given below, so you can try out that too.

In this example we set up an image classifier, where the user can upload an image and get the top 5 class predictions in return.

In [267]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.applications.mobilenet_v2 import (
    MobileNetV2,
    preprocess_input,
    decode_predictions,
)
from PIL import Image

## Set up a pretrained model

Let's download and set up a `MobileNetV2` model, trained on the 1000 classes in the ImageNet dataset. You can change this to anything you like. Remeber, however, to also use the appropriate preprocessing function.

In [268]:
# model = MobileNetV2(weights="imagenet")

# The model loading below is from Deep Learning with Python, Third edition. Chapter: 8, Fitting the model
model1 = keras.models.load_model(
    "../models/augment.keras"
)
# The model loading below is from Deep Learning with Python, Third edition. Chapter: 8, Fitting the model
model2 = keras.models.load_model(
    "../models/augment_no_rescale.keras"
)
# The model loading below is from Deep Learning with Python, Third edition. Chapter: 8, Fitting the model
model3 = keras.models.load_model(
    "../models/hierarchical_cnn_baseline_hybrid_crop.keras"
)

IMG_RES = (300, 300)

# Cellene under kommer tilnærmet direkte fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]

#### The two cells below are based on Deep Inside Convolutional Networks: Visualising Image Classification Models and Saliency Maps, Image-Specific Class Saliency Visualisation (https://arxiv.org/abs/1312.6034)

In [317]:
# This method is based on Deep Learning with Python, Third edition. Chapter: 2, Gradient computation,
# and get_gradients from https://keras.io/examples/vision/integrated_gradients/
def get_gradients(img_array, lvl, model):
    img_array /= 255.0
    with tf.GradientTape() as tape:
        tape.watch(img_array)
        # Denne veiledningen på reduce_max ble brukt: https://www.tensorflow.org/api_docs/python/tf/math/reduce_max
        print(img_array)
        score = tf.math.reduce_max(model(img_array)[lvl])
        print(score)
    gradients = tape.gradient(score, img_array)
    return gradients

In [318]:
def get_heatmap(img_array, lvl, model):
    gradients = get_gradients(img_array, lvl, model)
    abs_gradients = np.abs(gradients[0])
    # np.max is from: Deep Learning with Python, Third edition. Chapter: 10, Displaying the class activation heatmap
    max_gradient = np.max(abs_gradients)
    pixel_gradients = ((abs_gradients / max_gradient) * 255.0)

    values = 0
    heatmap = np.zeros(IMG_RES)

    for i in range(len(pixel_gradients)):
        for j in range(len(pixel_gradients[i])):
            # np.max is from: Deep Learning with Python, Third edition. Chapter: 10, Displaying the class activation heatmap
            heatmap[i][j] = np.max(pixel_gradients[i][j])
            values += heatmap[i][j]
    
    return heatmap

### The below cell is strongly based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation

In [319]:
import matplotlib.cm as cm

def superimpose(img_array, heatmap):
    # Uses the "jet" colormap to recolorize the heatmap
    jet = cm.get_cmap("jet")

    jet_colors = jet(np.arange(256))[:, :3]
    jet_colors-=[0,0,0.5]

    # Convertion to int is from: https://www.w3schools.com/python/numpy/numpy_data_types.asp (Converting Data Type on Existing Arrays)
    jet_heatmap = jet_colors[(np.round(heatmap)).astype('i')]

    # Superimposes the heatmap and the original image
    superimposed_img = jet_heatmap + img_array / 3.0
    superimposed_img = keras.utils.array_to_img(superimposed_img)
    return superimposed_img

# Cellene over kommer tilnærmet direkte fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]

In [320]:
import imageio
import gradio as gr

from preprocessing import preprocess_image_tensor

def classify_image(img: Image.Image, checkbox_group):

    # Resize to the input image to what MobileNet expects
    # img_resized = img.resize(IMG_RES)
    arr = np.array(img)

    # Check the color channels, so we can take both grayscale, RGB, and RGBA images as input.
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    elif arr.shape[-1] == 4:
        arr = arr[..., :3]

    arr = arr / 255.0

    # Linjen under er basert på: https://www.tensorflow.org/api_docs/python/tf/keras/ops/convert_to_tensor
    # arr = keras.ops.convert_to_tensor(arr, dtype="float32")

    arr = preprocess_image_tensor(arr, IMG_RES[0])
    # np.max is from: Deep Learning with Python, Third edition. Chapter: 10, Displaying the class activation heatmap
    arr = arr / np.max(arr)

    # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
    arr = keras.ops.expand_dims(arr, 0)

    print(np.max(arr[0]))
    print(np.min(arr[0]))
    # arr = preprocess_input(arr.astype("float32"))

    # Linjen under er delvis basert på: https://www.gradio.app/docs/gradio/checkboxgroup
    return_array = [gr.Image(),gr.Image(),gr.Label(),gr.Label(),gr.Image(),gr.Image(),gr.Label(),gr.Label(),gr.Image(),gr.Image(),gr.Label(),gr.Label(), img, gr.CheckboxGroup()]

    if "Augment.keras" in checkbox_group:
        model = model1
        # Predict!
        preds = model.predict(arr, verbose=0)

        # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
        heatmap = get_heatmap(arr, "lvl1", model)
        
        # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
        superimposed = superimpose(arr[0], heatmap)

        # Line below is based on: https://pillow.readthedocs.io/en/stable/reference/Image.html
        superimposed_lvl2 = Image.new(mode="RGB", size=IMG_RES)
        if preds["lvl1"][0][0] > 0.5:
            # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
            heatmap_lvl2 = get_heatmap(arr, "lvl2", model)
            # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
            superimposed_lvl2 = superimpose(arr[0], heatmap_lvl2)

        # Line below is based on https://stackoverflow.com/questions/57253048/scipy-misc-has-no-attribute-imsave and https://www.geeksforgeeks.org/python/getting-started-with-imageio-library-in-python/
        # imageio.imwrite("Neinieg.png", superimposed)

        car_model_preds = preds["lvl2"][0]
        return_array[0:4] = [superimposed, superimposed_lvl2, {"Tesla": preds["lvl1"][0][0], "Ikke-Tesla": 1-preds["lvl1"][0][0]}, {'3 2017–2023': car_model_preds[0], '3 2024–nå': car_model_preds[1], 'S 2012–2015': car_model_preds[2], 'S 2016–nå': car_model_preds[3], 'X': car_model_preds[4], 'Y 2020–2024': car_model_preds[5], 'Y 2025-nå': car_model_preds[6]}]

    if "Augment_no_rescale.keras" in checkbox_group:
        model = model2
        # Predict!
        preds = model.predict(arr, verbose=0)

        # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
        heatmap = get_heatmap(arr, "lvl1", model)
        
        # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
        superimposed = superimpose(arr[0], heatmap)

        # Line below is based on: https://pillow.readthedocs.io/en/stable/reference/Image.html
        superimposed_lvl2 = Image.new(mode="RGB", size=IMG_RES)
        if preds["lvl1"][0][0] > 0.5:
            # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
            heatmap_lvl2 = get_heatmap(arr, "lvl2", model)
            # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
            superimposed_lvl2 = superimpose(arr[0], heatmap_lvl2)

        # Line below is based on https://stackoverflow.com/questions/57253048/scipy-misc-has-no-attribute-imsave and https://www.geeksforgeeks.org/python/getting-started-with-imageio-library-in-python/
        # imageio.imwrite("Neinieg.png", superimposed)

        car_model_preds = preds["lvl2"][0]
        return_array[4:8] = [superimposed, superimposed_lvl2, {"Tesla": preds["lvl1"][0][0], "Ikke-Tesla": 1-preds["lvl1"][0][0]}, {'3 2017–2023': car_model_preds[0], '3 2024–nå': car_model_preds[1], 'S 2012–2015': car_model_preds[2], 'S 2016–nå': car_model_preds[3], 'X': car_model_preds[4], 'Y 2020–2024': car_model_preds[5], 'Y 2025-nå': car_model_preds[6]}]
    
    if "Hierarchical_cnn_baseline_hybrid_crop.keras" in checkbox_group:
        model = model3
        # Predict!
        preds = model.predict(arr, verbose=0)

        # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
        heatmap = get_heatmap(arr, "lvl1", model)
        
        # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
        superimposed = superimpose(arr[0], heatmap)

        # Line below is based on: https://pillow.readthedocs.io/en/stable/reference/Image.html
        superimposed_lvl2 = Image.new(mode="RGB", size=IMG_RES)
        if preds["lvl1"][0][0] > 0.5:
            # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
            heatmap_lvl2 = get_heatmap(arr, "lvl2", model)
            # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
            superimposed_lvl2 = superimpose(arr[0], heatmap_lvl2)
        # Line below is based on https://stackoverflow.com/questions/57253048/scipy-misc-has-no-attribute-imsave and https://www.geeksforgeeks.org/python/getting-started-with-imageio-library-in-python/
        # imageio.imwrite("Neinieg.png", superimposed)

        car_model_preds = preds["lvl2"][0]
        return_array[8:12] = [superimposed, superimposed_lvl2, {"Tesla": preds["lvl1"][0][0], "Ikke-Tesla": 1-preds["lvl1"][0][0]}, {'3 2017–2023': car_model_preds[0], '3 2024–nå': car_model_preds[1], 'S 2012–2015': car_model_preds[2], 'S 2016–nå': car_model_preds[3], 'X': car_model_preds[4], 'Y 2020–2024': car_model_preds[5], 'Y 2025-nå': car_model_preds[6]}]
    return return_array



#### Cellen under er basert på: https://github.com/gradio-app/gradio/issues/2066 og https://discuss.huggingface.co/t/how-to-programmatically-enable-or-disable-components/52350

In [321]:
def change_visibility(checkbox_group, model_1_label1, model_1_label2, model_2_label1, model_2_label2, model_3_label1, model_3_label2):
    return_array = [gr.Image(),gr.Image(),gr.Label(),gr.Label(),gr.Markdown(),gr.Image(),gr.Image(),gr.Label(),gr.Label(),gr.Markdown(),gr.Image(),gr.Image(),gr.Label(),gr.Label(),gr.Markdown()]
    if "Augment.keras" in checkbox_group:
        try:
            model_1_label1["Tesla"]
            # The line below is based on: https://www.geeksforgeeks.org/python/python-get-key-with-maximum-value-in-dictionary/
            predicted_model = max(model_1_label2, key=model_1_label2.get)
            return_array[0:5] = [gr.Image(label="Heatmap som viser hvorfor Tesla", visible=True), gr.Image(visible=True, label="Heatmap som viser hvorfor " + predicted_model), gr.Label(visible=True), gr.Label(visible=True), gr.Markdown(visible=True)]
        except:
            return_array[0:5] = [gr.Image(label="Heatmap som viser hvorfor ikke-Tesla", visible=True), gr.Image(visible=False), gr.Label(visible=True), gr.Label(visible=False), gr.Markdown(visible=True)]
    else: 
        return_array[0:5] = [gr.Image(visible=False), gr.Image(visible=False), gr.Label(visible=False), gr.Label(visible=False), gr.Markdown(visible=False)]
    if "Augment_no_rescale.keras" in checkbox_group:
        try:
            model_2_label1["Tesla"]
            # The line below is based on: https://www.geeksforgeeks.org/python/python-get-key-with-maximum-value-in-dictionary/
            predicted_model = max(model_2_label2, key=model_2_label2.get)
            return_array[5:10] = [gr.Image(label="Heatmap som viser hvorfor Tesla", visible=True), gr.Image(visible=True, label="Heatmap som viser hvorfor " + predicted_model), gr.Label(visible=True), gr.Label(visible=True), gr.Markdown(visible=True)]
        except:
            return_array[5:10] = [gr.Image(label="Heatmap som viser hvorfor ikke-Tesla", visible=True), gr.Image(visible=False), gr.Label(visible=True), gr.Label(visible=False), gr.Markdown(visible=True)]
    else: 
        return_array[5:10] = [gr.Image(visible=False), gr.Image(visible=False), gr.Label(visible=False), gr.Label(visible=False), gr.Markdown(visible=False)]
    if "Hierarchical_cnn_baseline_hybrid_crop.keras" in checkbox_group:
        try:
            model_3_label1["Tesla"]
            # The line below is based on: https://www.geeksforgeeks.org/python/python-get-key-with-maximum-value-in-dictionary/
            predicted_model = max(model_3_label2, key=model_3_label2.get)
            return_array[10:15] = [gr.Image(label="Heatmap som viser hvorfor Tesla", visible=True), gr.Image(visible=True, label="Heatmap som viser hvorfor " + predicted_model), gr.Label(visible=True), gr.Label(visible=True), gr.Markdown(visible=True)]
        except:
            return_array[10:15] = [gr.Image(label="Heatmap som viser hvorfor ikke-Tesla", visible=True), gr.Image(visible=False), gr.Label(visible=True), gr.Label(visible=False), gr.Markdown(visible=True)]
    else: 
        return_array[10:15] = [gr.Image(visible=False), gr.Image(visible=False), gr.Label(visible=False), gr.Label(visible=False), gr.Markdown(visible=False)]
    return return_array

## Set up the Gradio interface

Check the [documentation](https://www.gradio.app/docs) for the various things we can add here.

In [322]:
# Example images
examples = [
    "https://cdn.britannica.com/79/232779-050-6B0411D7/German-Shepherd-dog-Alsatian.jpg",
    "https://cdn.britannica.com/41/126641-050-E1CA0E61/cat-suns-hill-Parthenon-Athens-Greece-Acropolis.jpg",
]

with gr.Blocks(title="Multiattributt visuell kjøretøygjenkjenning under varierende lysforhold") as demo:
    gr.Markdown("## Multiattributt visuell kjøretøygjenkjenning under varierende lysforhold")
    gr.Markdown(
        "Last opp et bilde av et kjøretøy eller velg et eksempel under. "
    )
    with gr.Row():
        components = [[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],0,0]
        components[3] = gr.Image(type="pil", label="Opplastet bilde")
        # Linjen under er basert på: https://www.gradio.app/docs/gradio/checkboxgroup
        components[4] = gr.CheckboxGroup(choices=["Augment.keras", "Augment_no_rescale.keras", "Hierarchical_cnn_baseline_hybrid_crop.keras"])

    # Linjen under er basert på: https://www.gradio.app/docs/gradio/blocks
    components[0][4] = gr.Markdown("Prediksjonene til Augment.keras:", visible=False)
    with gr.Row():
        # De to linjene under er delvis basert på: https://github.com/gradio-app/gradio/issues/10394
        components[0][0] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
        components[0][1] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
        # Linjen under er basert på: https://www.gradio.app/4.44.1/guides/controlling-layout
        with gr.Column():
            components[0][2] = gr.Label(num_top_classes=1, label="Bilmerke", visible=False)
            components[0][3] = gr.Label(num_top_classes=3, label="Bilmodell", visible=False)

    # Linjen under er basert på: https://www.gradio.app/docs/gradio/blocks
    components[1][4] = gr.Markdown("Prediksjonene til Augment_no_rescale.keras:", visible=False)
    with gr.Row():
        # De to linjene under er delvis basert på: https://github.com/gradio-app/gradio/issues/10394
        components[1][0] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
        components[1][1] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
        # Linjen under er basert på: https://www.gradio.app/4.44.1/guides/controlling-layout
        with gr.Column():
            components[1][2] = gr.Label(num_top_classes=1, label="Bilmerke", visible=False)
            components[1][3] = gr.Label(num_top_classes=3, label="Bilmodell", visible=False)

    # Linjen under er basert på: https://www.gradio.app/docs/gradio/blocks
    components[2][4] = gr.Markdown("Prediksjonene til Hierarchical_cnn_baseline_hybrid_crop.keras:", visible=False)
    with gr.Row():
        # De to linjene under er delvis basert på: https://github.com/gradio-app/gradio/issues/10394
        components[2][0] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
        components[2][1] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
        # Linjen under er basert på: https://www.gradio.app/4.44.1/guides/controlling-layout
        with gr.Column():
            components[2][2] = gr.Label(num_top_classes=1, label="Bilmerke", visible=False)
            components[2][3] = gr.Label(num_top_classes=3, label="Bilmodell", visible=False)

    classify_btn = gr.Button("Classify", variant="primary")
    classify_btn.click(fn=classify_image, inputs=[components[3],components[4]], outputs=[components[0][0],components[0][1],components[0][2],components[0][3],components[1][0],components[1][1],components[1][2],components[1][3],components[2][0],components[2][1],components[2][2],components[2][3], components[3], components[4]])
    # Linjen under er basert på: https://github.com/gradio-app/gradio/issues/2066 og https://discuss.huggingface.co/t/how-to-programmatically-enable-or-disable-components/52350
    classify_btn.click(fn=change_visibility, inputs=[components[4], components[0][2], components[0][3], components[1][2], components[1][3], components[2][2], components[2][3]], outputs=[components[0][0], components[0][1], components[0][2], components[0][3], components[0][4], components[1][0], components[1][1], components[1][2], components[1][3], components[1][4], components[2][0], components[2][1], components[2][2], components[2][3], components[2][4]])

    examples = gr.Examples(
        examples=examples,
        inputs=components[3],
        # outputs=output,
        # fn=classify_image,
        # cache_examples=True
    )

Now we can run it:

In [ ]:
demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7906
* To create a public link, set `share=True` in `launch()`.


1.0
0.0
tf.Tensor(
[[[[0.00152494 0.00150955 0.00163264]
   [0.00150521 0.00148962 0.00161178]
   [0.00101847 0.00094104 0.00105113]
   ...
   [0.00294844 0.00332599 0.00326681]
   [0.00294709 0.00332227 0.00326388]
   [0.0029487  0.00331461 0.00325931]]

  [[0.00151372 0.00149833 0.00162142]
   [0.00147732 0.00146169 0.00158386]
   [0.00110163 0.0010261  0.00113847]
   ...
   [0.00297428 0.00332769 0.00326412]
   [0.00295376 0.00332646 0.00325725]
   [0.002951   0.00332251 0.00325353]]

  [[0.00137157 0.00135618 0.00147898]
   [0.0013687  0.0013538  0.00147571]
   [0.00116112 0.00111706 0.00122632]
   ...
   [0.00298571 0.00333908 0.00324859]
   [0.00297992 0.00333895 0.00324716]
   [0.00297657 0.00333496 0.00324319]]

  ...

  [[0.00146313 0.00181942 0.00206805]
   [0.00124546 0.00159745 0.00184782]
   [0.00133673 0.00166979 0.00192963]
   ...
   [0.00153502 0.0018889  0.00216584]
   [0.00148939 0.00184327 0.00212022]
   [0.00158802 0.00194189 0.00221884]]

  [[0.00168686 0.0020603  

C:\Users\johan\AppData\Local\Temp\ipykernel_4208\2727390636.py:5: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  jet = cm.get_cmap("jet")


tf.Tensor(
[[[[0.00152494 0.00150955 0.00163264]
   [0.00150521 0.00148962 0.00161178]
   [0.00101847 0.00094104 0.00105113]
   ...
   [0.00294844 0.00332599 0.00326681]
   [0.00294709 0.00332227 0.00326388]
   [0.0029487  0.00331461 0.00325931]]

  [[0.00151372 0.00149833 0.00162142]
   [0.00147732 0.00146169 0.00158386]
   [0.00110163 0.0010261  0.00113847]
   ...
   [0.00297428 0.00332769 0.00326412]
   [0.00295376 0.00332646 0.00325725]
   [0.002951   0.00332251 0.00325353]]

  [[0.00137157 0.00135618 0.00147898]
   [0.0013687  0.0013538  0.00147571]
   [0.00116112 0.00111706 0.00122632]
   ...
   [0.00298571 0.00333908 0.00324859]
   [0.00297992 0.00333895 0.00324716]
   [0.00297657 0.00333496 0.00324319]]

  ...

  [[0.00146313 0.00181942 0.00206805]
   [0.00124546 0.00159745 0.00184782]
   [0.00133673 0.00166979 0.00192963]
   ...
   [0.00153502 0.0018889  0.00216584]
   [0.00148939 0.00184327 0.00212022]
   [0.00158802 0.00194189 0.00221884]]

  [[0.00168686 0.0020603  0.002326